# Análise de Inadimplência e Performance Financeira
Análise exploratória de uma carteira de crédito simulada com foco em inadimplência, aging de parcelas e comportamento de pagamento por segmento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

## 1. Carregamento dos Dados

In [ ]:
clientes  = pd.read_csv('dados_clientes.csv', parse_dates=['data_cadastro'])
contratos = pd.read_csv('dados_contratos.csv', parse_dates=['data_inicio', 'data_vencimento'])
pagamentos = pd.read_csv('dados_pagamentos.csv', parse_dates=['data_vencimento', 'data_pagamento'])

print('Clientes: ', clientes.shape)
print('Contratos:', contratos.shape)
print('Pagamentos:', pagamentos.shape)

In [ ]:
clientes.head()

In [ ]:
contratos.head()

In [ ]:
pagamentos.head()

## 2. Limpeza e Enriquecimento

In [ ]:
# Merge das 3 tabelas
df = (pagamentos
      .merge(contratos, on='contrato_id')
      .merge(clientes,  on='cliente_id'))

# Recalcula dias_atraso para parcelas sem pagamento
df['dias_atraso'] = df.apply(
    lambda r: (r['data_pagamento'] - r['data_vencimento']).days
              if pd.notna(r['data_pagamento'])
              else (pd.Timestamp.today() - r['data_vencimento']).days,
    axis=1
)

# Situação de pagamento
def situacao(row):
    if pd.isna(row['data_pagamento']):
        return 'Não pago'
    elif row['dias_atraso'] > 0:
        return 'Atrasado'
    elif row['dias_atraso'] == 0:
        return 'Em dia'
    else:
        return 'Antecipado'

df['situacao_pagamento'] = df.apply(situacao, axis=1)

# Faixa de atraso (aging)
def faixa_atraso(dias):
    if dias <= 0:   return 'Sem atraso'
    elif dias <= 30: return '01-30 dias'
    elif dias <= 60: return '31-60 dias'
    elif dias <= 90: return '61-90 dias'
    else:            return '90+ dias'

df['faixa_atraso'] = df['dias_atraso'].apply(faixa_atraso)
df['valor_pago']   = df['valor_pago'].fillna(0)

print('Shape final:', df.shape)
df[['nome', 'produto', 'valor_devido', 'valor_pago', 'dias_atraso', 'situacao_pagamento']].head(8)

## 3. Visão Geral da Carteira

In [ ]:
total_contratos      = contratos.shape[0]
contratos_inadimp    = contratos[contratos['status'] == 'Inadimplente'].shape[0]
taxa_inadimplencia   = contratos_inadimp / total_contratos * 100
valor_em_risco       = df[df['situacao_pagamento'] == 'Não pago']['valor_devido'].sum()
ticket_medio         = contratos['valor_total'].mean()
score_medio          = clientes['score_credito'].mean()

print(f'Total de contratos      : {total_contratos}')
print(f'Contratos inadimplentes : {contratos_inadimp}')
print(f'Taxa de inadimplência   : {taxa_inadimplencia:.1f}%')
print(f'Valor em risco          : R$ {valor_em_risco:,.2f}')
print(f'Ticket médio            : R$ {ticket_medio:,.2f}')
print(f'Score médio da carteira : {score_medio:.0f}')

## 4. Status dos Contratos

In [ ]:
status_counts = contratos['status'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
cores = {'Ativo': '#1a5f9e', 'Quitado': '#1d9e75', 'Inadimplente': '#e24b4a', 'Renegociado': '#e07b39'}
colors = [cores.get(s, '#888') for s in status_counts.index]
axes[0].bar(status_counts.index, status_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Quantidade de Contratos por Status', fontsize=12)
axes[0].set_ylabel('Quantidade')
for i, v in enumerate(status_counts.values):
    axes[0].text(i, v + 0.05, str(v), ha='center', fontweight='bold')

# Pizza
axes[1].pie(status_counts.values, labels=status_counts.index,
            colors=colors, autopct='%1.0f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Distribuição de Status (%)', fontsize=12)

plt.tight_layout()
plt.show()

## 5. Inadimplência por Produto

In [ ]:
inadimp_produto = (contratos
    .groupby('produto')
    .apply(lambda x: pd.Series({
        'total': len(x),
        'inadimplentes': (x['status'] == 'Inadimplente').sum(),
        'taxa_pct': (x['status'] == 'Inadimplente').sum() / len(x) * 100
    }))
    .reset_index()
    .sort_values('taxa_pct', ascending=False))

print(inadimp_produto.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ['#e24b4a' if t == 100 else '#e07b39' if t > 0 else '#1d9e75'
              for t in inadimp_produto['taxa_pct']]
bars = ax.barh(inadimp_produto['produto'], inadimp_produto['taxa_pct'],
               color=bar_colors, edgecolor='white')
ax.set_xlabel('Taxa de Inadimplência (%)')
ax.set_title('Taxa de Inadimplência por Produto', fontsize=13)
ax.set_xlim(0, 115)
for bar, val in zip(bars, inadimp_produto['taxa_pct']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Aging — Distribuição de Atraso

In [ ]:
ordem_faixas = ['Sem atraso', '01-30 dias', '31-60 dias', '61-90 dias', '90+ dias']

aging = (df.groupby('faixa_atraso')
           .agg(qtd_parcelas=('pagamento_id', 'count'),
                valor_em_aberto=('valor_devido', lambda x: (x - df.loc[x.index, 'valor_pago']).sum()))
           .reindex(ordem_faixas)
           .fillna(0)
           .reset_index())

print(aging.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

cores_aging = ['#1d9e75', '#63c29a', '#e07b39', '#c05020', '#e24b4a']

axes[0].bar(aging['faixa_atraso'], aging['qtd_parcelas'],
            color=cores_aging, edgecolor='white')
axes[0].set_title('Quantidade de Parcelas por Faixa de Atraso', fontsize=11)
axes[0].set_ylabel('Parcelas')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(aging['faixa_atraso'], aging['valor_em_aberto'],
            color=cores_aging, edgecolor='white')
axes[1].set_title('Valor em Aberto por Faixa de Atraso (R$)', fontsize=11)
axes[1].set_ylabel('R$')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## 7. PF vs PJ — Risco e Volume

In [ ]:
seg = (df.merge(contratos[['contrato_id','valor_total']], on='contrato_id', suffixes=('','_ct'))
         .groupby('segmento')
         .agg(
             total_contratos  = ('contrato_id', 'nunique'),
             volume_carteira  = ('valor_total_ct', lambda x: contratos.loc[contratos['contrato_id'].isin(df.loc[x.index,'contrato_id']), 'valor_total'].drop_duplicates().sum()),
             valor_em_risco   = ('valor_devido',   lambda x: x[df.loc[x.index,'situacao_pagamento']=='Não pago'].sum()),
             score_medio      = ('score_credito',  'mean')
         )
         .reset_index())

# Forma simplificada usando merge direto
seg2 = (contratos
        .merge(clientes[['cliente_id','segmento','score_credito']], on='cliente_id')
        .groupby('segmento')
        .agg(
            total_contratos = ('contrato_id','count'),
            volume_carteira = ('valor_total','sum'),
            ticket_medio    = ('valor_total','mean'),
            score_medio     = ('score_credito','mean')
        )
        .reset_index())

print(seg2.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
cores_seg = ['#1a5f9e', '#e07b39']

axes[0].bar(seg2['segmento'], seg2['volume_carteira'], color=cores_seg, edgecolor='white')
axes[0].set_title('Volume Total da Carteira')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1000:.0f}k'))

axes[1].bar(seg2['segmento'], seg2['ticket_medio'], color=cores_seg, edgecolor='white')
axes[1].set_title('Ticket Médio por Contrato')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1000:.0f}k'))

axes[2].bar(seg2['segmento'], seg2['score_medio'], color=cores_seg, edgecolor='white')
axes[2].set_title('Score Médio de Crédito')
axes[2].set_ylim(0, 1000)
axes[2].axhline(600, color='gray', linestyle='--', linewidth=1, label='Meta: 600')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 8. Score de Crédito x Inadimplência

In [ ]:
def faixa_score(s):
    if s < 300:   return 'Baixo (0-299)'
    elif s < 600: return 'Regular (300-599)'
    elif s < 800: return 'Bom (600-799)'
    else:         return 'Excelente (800+)'

df['faixa_score'] = df['score_credito'].apply(faixa_score)

score_atraso = (df.groupby('faixa_score')
    .apply(lambda x: pd.Series({
        'total_parcelas':   len(x),
        'parcelas_atraso':  (x['dias_atraso'] > 0).sum(),
        'pct_atraso':       (x['dias_atraso'] > 0).sum() / len(x) * 100
    }))
    .reset_index()
    .sort_values('pct_atraso', ascending=False))

print(score_atraso.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

cores_score = ['#e24b4a', '#e07b39', '#1a5f9e', '#1d9e75']
axes[0].bar(score_atraso['faixa_score'], score_atraso['pct_atraso'],
            color=cores_score[:len(score_atraso)], edgecolor='white')
axes[0].set_title('% de Parcelas em Atraso por Faixa de Score', fontsize=11)
axes[0].set_ylabel('% em atraso')
axes[0].tick_params(axis='x', rotation=12)
for i, v in enumerate(score_atraso['pct_atraso']):
    axes[0].text(i, v + 1, f'{v:.0f}%', ha='center', fontweight='bold', fontsize=10)

# Dispersão score x dias_atraso
cores_sit = {'Em dia': '#1d9e75', 'Atrasado': '#e07b39',
             'Não pago': '#e24b4a', 'Antecipado': '#1a5f9e'}
for sit, grp in df.groupby('situacao_pagamento'):
    axes[1].scatter(grp['score_credito'], grp['dias_atraso'],
                    label=sit, color=cores_sit.get(sit,'#888'),
                    alpha=0.8, s=80, edgecolors='white', linewidths=0.5)
axes[1].set_title('Score de Crédito vs Dias de Atraso', fontsize=11)
axes[1].set_xlabel('Score de Crédito')
axes[1].set_ylabel('Dias de Atraso')
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 9. Correlação entre Variáveis Numéricas

In [ ]:
num_cols = df[['score_credito', 'valor_devido', 'valor_pago', 'dias_atraso']].copy()
corr = num_cols.corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Mapa de Correlação', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Principais Descobertas

In [ ]:
print('=' * 55)
print('PRINCIPAIS DESCOBERTAS')
print('=' * 55)

exp_pj = df[df['segmento']=='PJ'][df['situacao_pagamento']=='Não pago']['valor_devido'].sum()
exp_total = df[df['situacao_pagamento']=='Não pago']['valor_devido'].sum()
pct_pj = exp_pj / exp_total * 100 if exp_total > 0 else 0

criticos = df[df['faixa_atraso']=='90+ dias']
pct_score_baixo = (criticos['score_credito'] < 500).sum() / len(criticos) * 100 if len(criticos) > 0 else 0

print(f'\n1. Concentração de risco em PJ')
print(f'   PJ representa {pct_pj:.0f}% do valor inadimplente total')

print(f'\n2. Score como preditor de inadimplência')
print(f'   {pct_score_baixo:.0f}% das parcelas críticas (90+ dias) são de clientes com score < 500')

print(f'\n3. Consignado: produto mais saudável')
print(f'   Taxa de inadimplência: 0% — padrão esperado pelo desconto em folha')

print(f'\n4. Capital de Giro: maior risco')
print(f'   Taxa de inadimplência: 100% — concentra todo o valor em aberto crítico')
print('=' * 55)